In [5]:
import requests
from bs4 import BeautifulSoup

url = "http://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

In [6]:
books = soup.find_all("article", class_="product_pod")

data = []
for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    availability = book.find("p", class_="instock availability").text.strip()
    rating = book.p["class"][1]  # e.g. "Three"
    data.append({
        "title": title,
        "price": price,
        "availability": availability,
        "rating": rating
    })

In [7]:
import time

all_data = []
for page in range(1, 51):  # site has 50 pages
    url = f"http://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")
    for book in books:
        all_data.append({
            "title": book.h3.a["title"],
            "price": book.find("p", class_="price_color").text,
            "rating": book.p["class"][1]
        })
    time.sleep(1)  # be polite, don't overload the server

In [8]:
import pandas as pd

df = pd.DataFrame(all_data)
df.to_csv("books_dataset.csv", index=False)
print(df.head())

                                   title    price rating
0                   A Light in the Attic  Â£51.77  Three
1                     Tipping the Velvet  Â£53.74    One
2                             Soumission  Â£50.10    One
3                          Sharp Objects  Â£47.82   Four
4  Sapiens: A Brief History of Humankind  Â£54.23   Five


In [9]:
df['price'] = df['price'].str.replace('Â£', '£', regex=False)
df.to_csv("books_dataset.csv", index=False)  # re-save with fix
print(df.head())

                                   title   price rating
0                   A Light in the Attic  £51.77  Three
1                     Tipping the Velvet  £53.74    One
2                             Soumission  £50.10    One
3                          Sharp Objects  £47.82   Four
4  Sapiens: A Brief History of Humankind  £54.23   Five


In [10]:
print(df.shape)  # should show (1000, 3) or close to it

(1000, 3)


In [11]:
print(df.isnull().sum())
print(df.duplicated().sum())

title     0
price     0
rating    0
dtype: int64
0


In [12]:
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df['rating'] = df['rating'].map(rating_map)

In [13]:
df['price'] = df['price'].str.replace('£', '', regex=False).astype(float)

In [14]:
df.to_csv("books_dataset_clean.csv", index=False)

In [15]:
pd.read_csv('books_dataset_clean.csv').head() 

,title,price,rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5
